# Data Dharma by Srikanth
## SQL ↔ PySpark Bridge — Part 5
### Window Functions: ROW_NUMBER, RANK, DENSE_RANK, Aggregate Windows, Running Totals, LAG & LEAD

**PART 1 — FOUNDATIONS ✅**
SELECT, FILTER, DISTINCT, SORT, LIMIT

**PART 2 — TRANSFORMATIONS ✅**
CASE WHEN, CAST, IN, LIKE, NULL, String & Date Functions

**PART 3 — AGGREGATIONS ✅**
GROUP BY, COUNT, SUM, AVG, MIN, MAX, HAVING

**PART 4 — JOINS ✅**
INNER, LEFT, RIGHT, FULL OUTER JOIN

**PART 5 — WINDOW FUNCTIONS 🚀**
ROW_NUMBER, RANK, DENSE_RANK, Aggregate Windows, Running Total, LAG, LEAD

Part 3 taught us how to summarize groups using `GROUP BY`. But sometimes we need a calculation across related rows **without losing the original rows**. That's where Window Functions help.

## SECTION 2 — GROUP BY vs WINDOW FUNCTION

Suppose we want the total order amount for each city.

```
GROUP BY city
    ↓
ONE row per city
(individual orders collapse away)

WINDOW FUNCTION
    ↓
EVERY order row stays
(the city total is added alongside each one)
```

**Memory:**
`GROUP BY` → rows **collapse**
`WINDOW` → rows **remain**

We'll prove this with real numbers in Section 8.

## SECTION 3 — WINDOW FUNCTION ANATOMY

When building a window expression, think about up to three questions:

```
PARTITION BY   → Which group?
ORDER BY       → What order inside the group?
FUNCTION       → What calculation or ranking?
```

**SQL mental model:**
```
FUNCTION() OVER (
    PARTITION BY ...
    ORDER BY ...
)
```

**PySpark mental model:**
```
Window.partitionBy(...).orderBy(...)
```
then:
```
.withColumn(
    ...,
    function().over(window_spec)
)
```

**Important:** `ORDER BY` isn't mandatory for every window function. Ranking functions (`ROW_NUMBER`, `RANK`, `DENSE_RANK`) require it — there's no "rank" without an order. But an aggregate window calculation, like a city total, doesn't need ordering at all — Section 8 will show exactly that.

## SECTION 0 — Data Setup

Reusing the exact same `orders` dataset from Parts 1–4 — same rows, same `order_amount` formula. This notebook is independently runnable — you don't need to run earlier parts first.

In [0]:
# from datetime is a module in Python and date is a class in the datetime module
from datetime import date

# from pyspark.sql.types is a module in Python and StructType is a class in it
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    DoubleType,
)

# from pyspark.sql.window is a module in Python and Window is a class in it
from pyspark.sql.window import Window

# from pyspark.sql.functions is a module in Python
# each of these is a function in the pyspark.sql.functions module
from pyspark.sql.functions import (
    col,
    row_number,
    rank,
    dense_rank,
    sum as spark_sum,
    avg,
    lag,
    lead,
    round as spark_round,
)

In [0]:
# below is the schema for the orders table
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("order_date", DateType(), False),
    StructField("ship_date", DateType(), True),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("payment_method", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount_amount", DoubleType(), False),
    StructField("promo_code", StringType(), True),
])

In [0]:
# below is the list with sample data
orders_data = [
    (1001, 501, "  raj kumar",      date(2026, 1, 5),  date(2026, 1, 8),  "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 3, 250.00, 20.00, "SAVE10"),
    (1002, 502, "MEENA REDDY ",     date(2026, 1, 6),  date(2026, 1, 8),  "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  2, 150.00, 10.00, None),
    (1003, 503, "Suresh Babu",      date(2026, 1, 7),  None,              "Austin",   "TX", "PENDING",   "PAYPAL",      1, 500.00,  0.00, "WELCOME5"),
    (1004, 504, "Anita Rao",        date(2026, 1, 8),  date(2026, 1, 12), "Chicago",  "IL", "COMPLETED", "UPI",         5,  80.00, 15.00, None),
    (1005, 505, "Kiran Varma",      date(2026, 1, 9),  None,              "New York", "NY", "CANCELLED", "CREDIT_CARD", 4, 120.00,  0.00, None),
    (1006, 501, " Raj Kumar",       date(2026, 1, 10), date(2026, 1, 13), "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 2, 300.00, 25.00, "SAVE10"),
    (1007, 506, "Divya Nair",       date(2026, 1, 11), date(2026, 1, 13), "Dallas",   "TX", "RETURNED",  "DEBIT_CARD",  1, 200.00,  0.00, None),
    (1008, 507, "ramesh iyer",      date(2026, 1, 12), date(2026, 1, 17), "Austin",   "TX", "COMPLETED", "PAYPAL",      3, 100.00, 10.00, "FESTIVE20"),
    (1009, 508, "Priya Sharma",     date(2026, 1, 13), None,              "Chicago",  "IL", "PENDING",   "UPI",         2,  60.00,  5.00, None),
    (1010, 509, "Arjun Menon",      date(2026, 1, 14), date(2026, 1, 17), "New York", "NY", "COMPLETED", "CREDIT_CARD", 6,  90.00, 20.00, "WELCOME5"),
    (1011, 502, "MEENA REDDY",      date(2026, 1, 15), date(2026, 1, 17), "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  4, 175.00, 30.00, None),
    (1012, 510, "Lakshmi Pillai",   date(2026, 1, 16), date(2026, 1, 22), "Houston",  "TX", "COMPLETED", "PAYPAL",      1, 800.00, 50.00, "SAVE10"),
    (1013, 511, "Vikram Rao",       date(2026, 1, 17), None,              "Austin",   "TX", "CANCELLED", "UPI",         2, 250.00,  0.00, None),
    (1014, 512, "Sneha Gupta",      date(2026, 1, 18), date(2026, 1, 22), "Chicago",  "IL", "COMPLETED", "CREDIT_CARD", 3, 220.00, 10.00, "FESTIVE20"),
    (1015, 513, "Karthik Reddy",    date(2026, 1, 19), date(2026, 1, 22), "New York", "NY", "RETURNED",  "DEBIT_CARD",  1, 400.00,  0.00, None),
    (1016, 503, "  Suresh Babu  ",  date(2026, 1, 20), date(2026, 1, 22), "Austin",   "TX", "COMPLETED", "PAYPAL",      5,  60.00,  5.00, "WELCOME5"),
    (1017, 514, "Deepa Krishnan",   date(2026, 1, 21), None,              "Houston",  "TX", "PENDING",   "CREDIT_CARD", 2, 175.00,  0.00, None),
    (1018, 505, "Kiran Varma",      date(2026, 1, 22), date(2026, 1, 25), "New York", "NY", "COMPLETED", "UPI",         3, 210.00, 15.00, "SAVE10"),
]

# create a dataframe with the schema and data
orders_raw_df = spark.createDataFrame(orders_data, schema=orders_schema)

In [0]:
# create a new column 'order_amount' by multiplying 'quantity' and 'unit_price' and subtracting 'discount_amount'
# round the result to 2 decimal places and cast it to decimal(10,2)

orders_df = orders_raw_df.withColumn(
    "order_amount",
    spark_round((col("quantity") * col("unit_price")) - col("discount_amount"), 2).cast("decimal(10,2)")
)

# display the dataframe
display(orders_df)

In [0]:
# create a temp view 'orders' from the dataframe
orders_df.createOrReplaceTempView("orders")

From this point onward, SQL and PySpark are reading the same data —
SQL through the view `orders`, PySpark through the DataFrame `orders_df`.

## SECTION 4 — ROW_NUMBER

**Business Requirement:** Number orders within each customer from newest to oldest.

### 🟨 SQL

In [0]:
%sql
-- Number orders within each customer from newest to oldest

SELECT
    order_id,
    customer_id,
    order_date,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY order_date DESC, order_id DESC
    ) AS order_seq
FROM orders
ORDER BY customer_id, order_seq;

### 🟦 PySpark

In [0]:
# Number orders within each customer from newest to oldest

customer_order_window = Window.partitionBy("customer_id").orderBy(
    col("order_date").desc(),
    col("order_id").desc()
)

result_df = (
    orders_df
    .select(
        "order_id",
        "customer_id",
        "order_date",
        row_number().over(customer_order_window).alias("order_seq")
    )
    .orderBy("customer_id", "order_seq")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` ↔ `row_number().over(Window.partitionBy(...).orderBy(...))`

`ROW_NUMBER` always assigns unique sequential numbers — `1, 2, 3, ...` — even when the ordering values tie. The secondary `order_id DESC` in the `ORDER BY` matters here: without it, two orders on the same date could be numbered in either order, and the result wouldn't be deterministic.

## SECTION 5 — RANK

**Business Requirement:** Rank orders by order amount within each city.

Austin has two orders tied at exactly `500.00` — a real tie, not a manufactured one — so we can see exactly how `RANK` handles it.

### 🟨 SQL

In [0]:
%sql
-- Rank orders by order amount within each city

SELECT
    order_id,
    city,
    order_amount,
    RANK() OVER (
        PARTITION BY city
        ORDER BY order_amount DESC
    ) AS amount_rank
FROM orders
WHERE city = 'Austin'
ORDER BY amount_rank;

### 🟦 PySpark

In [0]:
# Rank orders by order amount within each city

city_amount_window = Window.partitionBy("city").orderBy(col("order_amount").desc())

result_df = (
    orders_df
    .filter(col("city") == "Austin")
    .select(
        "order_id",
        "city",
        "order_amount",
        rank().over(city_amount_window).alias("amount_rank")
    )
    .orderBy("amount_rank")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`RANK() OVER (...)` ↔ `rank().over(...)`

Both orders at `500.00` get rank `1`. The next order, at `295.00`, gets rank `3` — `RANK` **skips** the number `2` because two rows already took rank `1`.

## SECTION 6 — DENSE_RANK

**Business Requirement:** Same as Section 5 — rank Austin's orders by order amount — but using `DENSE_RANK` instead.

Same data, same tie, so we can compare directly.

### 🟨 SQL

In [0]:
%sql
-- Rank orders by order amount within each city, without gaps after ties

SELECT
    order_id,
    city,
    order_amount,
    DENSE_RANK() OVER (
        PARTITION BY city
        ORDER BY order_amount DESC
    ) AS amount_dense_rank
FROM orders
WHERE city = 'Austin'
ORDER BY amount_dense_rank;

### 🟦 PySpark

In [0]:
# Rank orders by order amount within each city, without gaps after ties

result_df = (
    orders_df
    .filter(col("city") == "Austin")
    .select(
        "order_id",
        "city",
        "order_amount",
        dense_rank().over(city_amount_window).alias("amount_dense_rank")
    )
    .orderBy("amount_dense_rank")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`DENSE_RANK() OVER (...)` ↔ `dense_rank().over(...)`

Both orders at `500.00` still get rank `1`. But now the `295.00` order gets rank `2` — `DENSE_RANK` does **not** leave a gap after a tie.

**Memory:**
`ROW_NUMBER` → always a unique sequence
`RANK` → ties share a rank; gaps possible after
`DENSE_RANK` → ties share a rank; no gaps after

## SECTION 7 — ROW_NUMBER vs RANK vs DENSE_RANK

One result, all three side by side — the difference should be obvious at a glance.

### 🟨 SQL

In [0]:
%sql
-- Compare ROW_NUMBER, RANK, and DENSE_RANK on the same tied data

SELECT
    order_id,
    city,
    order_amount,
    ROW_NUMBER() OVER (PARTITION BY city ORDER BY order_amount DESC, order_id ASC) AS row_num,
    RANK() OVER (PARTITION BY city ORDER BY order_amount DESC) AS rnk,
    DENSE_RANK() OVER (PARTITION BY city ORDER BY order_amount DESC) AS dense_rnk
FROM orders
WHERE city = 'Austin'
ORDER BY order_amount DESC, order_id ASC;

### 🟦 PySpark

In [0]:
# Compare ROW_NUMBER, RANK, and DENSE_RANK on the same tied data

city_amount_window_rn = Window.partitionBy("city").orderBy(
    col("order_amount").desc(),
    col("order_id").asc()
)

result_df = (
    orders_df
    .filter(col("city") == "Austin")
    .select(
        "order_id",
        "city",
        "order_amount",
        row_number().over(city_amount_window_rn).alias("row_num"),
        rank().over(city_amount_window).alias("rnk"),
        dense_rank().over(city_amount_window).alias("dense_rnk")
    )
    .orderBy(col("order_amount").desc(), col("order_id").asc())
)

# display data in the DataFrame result_df
display(result_df)

`ROW_NUMBER` needs its own tiebreak (`order_id ASC`) to stay deterministic — `RANK` and `DENSE_RANK` don't, since tied rows are supposed to share the same value either way.

## SECTION 8 — AGGREGATE WINDOW FUNCTION

**Business Requirement:** Show each individual order AND the total order amount for its city.

This is where the `GROUP BY` vs `WINDOW` distinction from Section 2 becomes real. A `GROUP BY` here would return one row per city. A window `SUM` returns the city total **alongside every individual order**.

### 🟨 SQL

In [0]:
%sql
-- Show each order along with the total order amount for its city

SELECT
    order_id,
    city,
    order_amount,
    SUM(order_amount) OVER (PARTITION BY city) AS city_total
FROM orders
ORDER BY city, order_id;

### 🟦 PySpark

In [0]:
# Show each order along with the total order amount for its city

city_total_window = Window.partitionBy("city")

result_df = (
    orders_df
    .select(
        "order_id",
        "city",
        "order_amount",
        spark_sum("order_amount").over(city_total_window).alias("city_total")
    )
    .orderBy("city", "order_id")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`SUM(order_amount) OVER (PARTITION BY city)` ↔ `spark_sum("order_amount").over(Window.partitionBy("city"))`

Notice: no `ORDER BY` inside this window at all. This aggregate doesn't need ordering — it just needs to know *which group* to total, which is exactly the distinction from Section 3.

## SECTION 9 — AVG AS A WINDOW FUNCTION

**Business Requirement:** Show each order together with the average order amount for its city.

### 🟨 SQL

In [0]:
%sql
-- Show each order along with the average order amount for its city

SELECT
    order_id,
    city,
    order_amount,
    CAST(ROUND(AVG(order_amount) OVER (PARTITION BY city), 2) AS DECIMAL(10,2)) AS city_avg
FROM orders
ORDER BY city, order_id;

### 🟦 PySpark

In [0]:
# Show each order along with the average order amount for its city

result_df = (
    orders_df
    .select(
        "order_id",
        "city",
        "order_amount",
        spark_round(avg("order_amount").over(city_total_window), 2).cast("decimal(10,2)").alias("city_avg")
    )
    .orderBy("city", "order_id")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`AVG(order_amount) OVER (PARTITION BY city)` ↔ `avg("order_amount").over(Window.partitionBy("city"))`

Same `city_total_window` from Section 8, reused. This window spec defines the city partition, and multiple window functions can reuse it.

## SECTION 10 — RUNNING TOTAL

**Business Requirement:** Calculate the cumulative order amount for each customer, over time.

This is an **ordered** aggregate window — unlike Sections 8 and 9, order matters here, so we sort by `order_date` (with `order_id` as a deterministic tiebreak).

The window starts from the first row in that customer's ordered history and continues through the current row — that's what the explicit frame below says.

### 🟨 SQL

In [0]:
%sql
-- Calculate the cumulative order amount for each customer, over time

SELECT
    order_id,
    customer_id,
    order_date,
    order_amount,
    SUM(order_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date ASC, order_id ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM orders
ORDER BY customer_id, order_date;

### 🟦 PySpark

In [0]:
# Calculate the cumulative order amount for each customer, over time

customer_running_window = (
    Window.partitionBy("customer_id")
    .orderBy(col("order_date").asc(), col("order_id").asc())
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

result_df = (
    orders_df
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_amount",
        spark_sum("order_amount").over(customer_running_window).alias("running_total")
    )
    .orderBy("customer_id", "order_date")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` ↔ `.rowsBetween(Window.unboundedPreceding, Window.currentRow)`

Customer `501`'s first order (`730.00`) has a running total of `730.00`. Their second order (`575.00`) has a running total of `1305.00` — the frame included both rows up to and including the current one.

## SECTION 11 — LAG

**Business Requirement:** For each customer, show the previous order amount.

### 🟨 SQL

In [0]:
%sql
-- Show the previous order amount for each customer

SELECT
    order_id,
    customer_id,
    order_date,
    order_amount,
    LAG(order_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date ASC, order_id ASC
    ) AS prev_order_amount
FROM orders
ORDER BY customer_id, order_date;

### 🟦 PySpark

In [0]:
# Show the previous order amount for each customer

customer_order_window_asc = Window.partitionBy("customer_id").orderBy(
    col("order_date").asc(),
    col("order_id").asc()
)

result_df = (
    orders_df
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_amount",
        lag("order_amount").over(customer_order_window_asc).alias("prev_order_amount")
    )
    .orderBy("customer_id", "order_date")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`LAG(order_amount) OVER (...)` ↔ `lag("order_amount").over(...)`

`LAG` → the previous row's value. The **first** order in each customer's history has `prev_order_amount = NULL`, because there's no order before it.

## SECTION 12 — LEAD

**Business Requirement:** For each customer, show the next order amount.

### 🟨 SQL

In [0]:
%sql
-- Show the next order amount for each customer

SELECT
    order_id,
    customer_id,
    order_date,
    order_amount,
    LEAD(order_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date ASC, order_id ASC
    ) AS next_order_amount
FROM orders
ORDER BY customer_id, order_date;

### 🟦 PySpark

In [0]:
# Show the next order amount for each customer

result_df = (
    orders_df
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_amount",
        lead("order_amount").over(customer_order_window_asc).alias("next_order_amount")
    )
    .orderBy("customer_id", "order_date")
)

# display data in the DataFrame result_df
display(result_df)

**Key Mapping**

`LEAD(order_amount) OVER (...)` ↔ `lead("order_amount").over(...)`

`LEAD` → the next row's value. The **last** order in each customer's history has `next_order_amount = NULL`, because there's no order after it.

## SECTION 13 — REAL-WORLD COMBINED EXAMPLE

**Business Requirement:** For each customer, keep every individual order, rank it within that customer from highest to lowest amount, show the customer's total spend, their average order amount, and the previous order amount by date.

Four window calculations, one result, every original row preserved.

### 🟨 SQL

In [0]:
%sql
-- Show each order with its rank, customer total, customer average, and previous order amount

SELECT
    order_id,
    customer_id,
    order_date,
    order_amount,
    RANK() OVER (PARTITION BY customer_id ORDER BY order_amount DESC) AS amount_rank,
    SUM(order_amount) OVER (PARTITION BY customer_id) AS customer_total,
    CAST(ROUND(AVG(order_amount) OVER (PARTITION BY customer_id), 2) AS DECIMAL(10,2)) AS customer_avg,
    LAG(order_amount) OVER (PARTITION BY customer_id ORDER BY order_date ASC, order_id ASC) AS prev_order_amount
FROM orders
ORDER BY customer_id, order_date;

### 🟦 PySpark

In [0]:
# Show each order with its rank, customer total, customer average, and previous order amount

customer_rank_window = Window.partitionBy("customer_id").orderBy(col("order_amount").desc())
customer_total_window = Window.partitionBy("customer_id")

result_df = (
    orders_df
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_amount",
        rank().over(customer_rank_window).alias("amount_rank"),
        spark_sum("order_amount").over(customer_total_window).alias("customer_total"),
        spark_round(avg("order_amount").over(customer_total_window), 2).cast("decimal(10,2)").alias("customer_avg"),
        lag("order_amount").over(customer_order_window_asc).alias("prev_order_amount")
    )
    .orderBy("customer_id", "order_date")
)

# display data in the DataFrame result_df
display(result_df)

Same requirement. Same select-with-multiple-window-functions chain. Same result — expressed two ways.

Every window function here uses `customer_id` to partition, but each does a different job: `RANK` orders each customer's own orders, `SUM`/`AVG` summarize the whole customer, and `LAG` looks at chronological history. All of it coexists in one result, with every original row intact.

## SECTION 14 — VIEWER CHALLENGE

**Pause the video and try this first.**

**Requirement:** For `COMPLETED` orders only, for each state — keep every completed order, rank orders from highest `order_amount` to lowest, show the total completed order amount for that state, and the average completed order amount for that state.

1. Solve it in SQL.
2. Translate the same requirement into PySpark.

**Required output:** `order_id`, `state`, `order_amount`, `state_rank`, `state_total`, `state_avg`

--- PAUSE HERE ---

### 🟨 SQL SOLUTION

In [0]:
%sql
-- Rank completed orders within each state, with state totals and averages

SELECT
    order_id,
    state,
    order_amount,
    RANK() OVER (PARTITION BY state ORDER BY order_amount DESC) AS state_rank,
    SUM(order_amount) OVER (PARTITION BY state) AS state_total,
    CAST(ROUND(AVG(order_amount) OVER (PARTITION BY state), 2) AS DECIMAL(10,2)) AS state_avg
FROM orders
WHERE order_status = 'COMPLETED'
ORDER BY state, state_rank, order_id;

### 🟦 PYSPARK SOLUTION

In [0]:
# Rank completed orders within each state, with state totals and averages

state_rank_window = Window.partitionBy("state").orderBy(col("order_amount").desc())
state_total_window = Window.partitionBy("state")

result_df = (
    orders_df
    .filter(col("order_status") == "COMPLETED")
    .select(
        "order_id",
        "state",
        "order_amount",
        rank().over(state_rank_window).alias("state_rank"),
        spark_sum("order_amount").over(state_total_window).alias("state_total"),
        spark_round(avg("order_amount").over(state_total_window), 2).cast("decimal(10,2)").alias("state_avg")
    )
    .orderBy("state", "state_rank", "order_id")
)

# display data in the DataFrame result_df
display(result_df)

**RESULT:** 11 completed orders across 3 states. Watch Texas — two orders tie at `290.00` (`order_id` `1002` and `1008`) and both correctly land on `state_rank = 6`.

## Final Check — Did SQL and PySpark Return the Same Window Result?

We wrote the same window logic in SQL and PySpark, using the Section 14 challenge.

Each row has **6** selected business columns — `order_id`, `state`, `order_amount`, `state_rank`, `state_total`, and `state_avg`. So we compare the **complete row**, not just one identifier.

### What will we do?

1. Run the SQL version and store the result in `sql_result`
2. Run the PySpark version and store the result in `pyspark_result`
3. Both are sorted by `state, state_rank, order_id`, so tied ranks also have a deterministic output order
4. Collect each full row into a Python list
5. Compare the two lists directly

To be precise about what this verifies: for this business requirement and these selected output columns, both implementations produced the same business result — same data, same business requirement, different syntax, same result.

In [0]:
# Compare SQL and PySpark results to verify the window calculations match column by column

# Run the SQL query and store the result as a DataFrame
sql_result = spark.sql("""
    SELECT
        order_id,
        state,
        order_amount,
        RANK() OVER (PARTITION BY state ORDER BY order_amount DESC) AS state_rank,
        SUM(order_amount) OVER (PARTITION BY state) AS state_total,
        CAST(ROUND(AVG(order_amount) OVER (PARTITION BY state), 2) AS DECIMAL(10,2)) AS state_avg
    FROM orders
    WHERE order_status = 'COMPLETED'
    ORDER BY state, state_rank, order_id
""")

# Apply the same logic using PySpark and store the result as a DataFrame
pyspark_result = (
    orders_df
    .filter(col("order_status") == "COMPLETED")
    .select(
        "order_id",
        "state",
        "order_amount",
        rank().over(state_rank_window).alias("state_rank"),
        spark_sum("order_amount").over(state_total_window).alias("state_total"),
        spark_round(avg("order_amount").over(state_total_window), 2).cast("decimal(10,2)").alias("state_avg")
    )
    .orderBy("state", "state_rank", "order_id")
)

# Trigger execution with collect(), bring each full row to the driver, as a dictionary of column values
sql_rows = [row.asDict() for row in sql_result.collect()]

# Trigger execution with collect(), bring each full PySpark row to the driver, as a dictionary of column values
pyspark_rows = [row.asDict() for row in pyspark_result.collect()]

# Display both full result sets and check whether every column matches for every row
print("SQL rows:     ", sql_rows)
print("PySpark rows: ", pyspark_rows)
print("SQL and PySpark results match:", sql_rows == pyspark_rows)

### Understanding the Result

`row.asDict()` turns one Spark Row into a Python dictionary of `{column_name: value}` — comparing `Row` objects directly can be fragile, so we compare plain dictionaries instead.

`sql_rows == pyspark_rows` compares those lists directly. For this to be `True`, every row must appear in the same order, with the same values in all 6 selected columns — including the tied `state_rank = 6` on both Texas orders.

If the result is `True` ✅, our SQL and PySpark window logic produced identical output for the business columns we selected. This doesn't mean SQL and PySpark are the same language, or that they always generate identical physical execution plans — only that, for this requirement and these columns, they agree.

## SECTION 16 — FINAL CHEAT SHEET

| SQL | PySpark |
|---|---|
| `OVER (...)` | `.over(...)` |
| `PARTITION BY` | `Window.partitionBy(...)` |
| `ORDER BY` inside window | `.orderBy(...)` |
| `ROW_NUMBER()` | `row_number()` |
| `RANK()` | `rank()` |
| `DENSE_RANK()` | `dense_rank()` |
| `SUM() OVER(...)` | `spark_sum().over(...)` |
| `AVG() OVER(...)` | `avg().over(...)` |
| `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` | `.rowsBetween(Window.unboundedPreceding, Window.currentRow)` |
| `LAG()` | `lag()` |
| `LEAD()` | `lead()` |

**Memory, worth repeating:**

`GROUP BY` → rows **collapse**
`WINDOW` → rows **remain**

`ROW_NUMBER` → always unique
`RANK` → ties share rank, gaps possible
`DENSE_RANK` → ties share rank, no gaps

```
     SQL
      ↕
   PySpark
      ↓
Same Business Logic
```

## Part 5 complete — and with it, the SQL ↔ PySpark Bridge mini-series.

```
Part 1: SELECT, FILTER, DISTINCT, SORT, LIMIT
Part 2: CASE WHEN, CAST, IN, LIKE, NULL, String & Date Functions
Part 3: GROUP BY, COUNT, SUM, AVG, MIN, MAX, HAVING
Part 4: JOINs
Part 5: Window Functions
```

SAME DATA.
SAME BUSINESS REQUIREMENT.
DIFFERENT SYNTAX.
SAME RESULT.

If you've followed all five parts, you can now look at common SQL Data Engineering logic and know exactly how to structure the equivalent PySpark DataFrame code.